# W06｜NL2SQL / Semantic Layer / Guardrails｜教学版

**建议学习时间：6h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 复习下方映射前置后进入 NL2SQL Guardrails | 不新增泛学课程，围绕安全问数闭环 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Deterministic vs Agentic | 明确 LLM 与规则引擎边界 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 准备 10 条 Golden 问数问题 | 定义 metric/dimension/time 预期 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 实现 NL → Plan → SQL → PostgreSQL → Table | 至少 8/10 问题正确执行 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 列出 5 条不能交给 LLM 的规则 | 例如权限/row limit/危险 SQL | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜10 条 Golden
至少覆盖：monthly trend、channel compare、use_case ranking、cost trend、cost by channel/use_case 等，共 10 条，并显式写 expected metric/dimension/time。

### Task 4｜8/10 验收
对 10 条依次执行：Question → Plan → SQL → DB；统计成功数，`>=8` 才勾 Excel ChatBI Task。

### Task 5｜五条不能交给 LLM 的规则
1. Authorization；
2. Allowed tables/columns；
3. Metric formula；
4. Row/cost/timeout limit；
5. 禁止危险 SQL / read-only policy。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 构建受控 NL2SQL 链路
- 理解 semantic layer
- 建立第一批 golden cases

        ## 本周前置检查

        - 完成 W04-W05；会基本 SQL/Pydantic

        ## 本周必须记住的英文术语

        - `semantic layer`：语义层
- `logical plan`：逻辑计划
- `allowlist`：白名单
- `parameterized SQL`：参数化SQL
- `golden case`：黄金样例

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. 为什么不要让 LLM 直接任意生成并执行 SQL？

    **是什么（What）**  
    LLM 擅长理解模糊自然语言，但不应该决定权限、安全边界、允许字段、行数限制等。

    **为什么需要（Why）**  
    直接 SQL 容易产生权限风险、幻觉字段、超大查询、错误指标公式。

    **什么时候用（When）**  
    ChatBI / NL2SQL。

    **常见误区 / 容易踩坑**  
    “先让 LLM 写 SQL，再用一句 prompt 让它自检”仍然不等于 deterministic guardrail。


**中英术语**：`semantic layer` = 语义层；`logical plan` = 逻辑计划；`allowlist` = 白名单；`guardrail` = 护栏

### 代码实验：Question → QueryPlan → Safe SQL

**这段代码想让你看什么？**

观察：模型/Mock 只产生有限字段的 QueryPlan；真正 SQL 由程序从 allowlist 生成。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from pydantic import BaseModel
from typing import Literal
import sqlite3

class QueryPlan(BaseModel):
    metric: Literal["message_count", "cost"]
    dimension: Literal["month", "channel", "use_case"]
    limit: int = 100

METRICS = {
    "message_count": "SUM(volume)",
    "cost": "SUM(cost)",
}
DIMENSIONS = {
    "month": "month",
    "channel": "channel",
    "use_case": "use_case",
}

def build_sql(plan: QueryPlan):
    limit = min(max(plan.limit, 1), 100)
    metric = METRICS[plan.metric]
    dim = DIMENSIONS[plan.dimension]
    sql = f'''
    SELECT {dim}, {metric}
    FROM messages
    GROUP BY {dim}
    ORDER BY {dim}
    LIMIT ?
    '''
    return sql, (limit,)

### 代码实验：用 synthetic data 真正执行查询

**这段代码想让你看什么？**

重点不是 SQL 语法，而是验证 Plan 与结果能跑通。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE messages(month TEXT, channel TEXT, use_case TEXT, volume INTEGER, cost REAL)")
cur.executemany(
    "INSERT INTO messages VALUES (?, ?, ?, ?, ?)",
    [
        ("2026-01","SMS","UC_A",1000,120.0),
        ("2026-01","EMAIL","UC_A",800,40.0),
        ("2026-02","SMS","UC_A",1150,138.0),
        ("2026-02","EMAIL","UC_B",850,42.5),
        ("2026-03","SMS","UC_C",1210,145.2),
        ("2026-03","PUSH","UC_B",1620,17.0),
    ]
)

plan = QueryPlan(metric="message_count", dimension="month")
sql, params = build_sql(plan)
print(sql)
print(cur.execute(sql, params).fetchall())

## 2. Golden Case 为什么重要？

    **是什么（What）**  
    Golden Case 是你事先知道正确语义/结果的测试样例，用来衡量版本变更是否真正改善。

    **为什么需要（Why）**  
    AI 系统不能只靠“我感觉回答不错”。

    **什么时候用（When）**  
    Intent、metric、dimension、filter、SQL、answer 等分层评估。

    **常见误区 / 容易踩坑**  
    只看最终自然语言答案会掩盖中间 plan 错误。


**中英术语**：`golden dataset` = 黄金测试集；`evaluation` = 评估；`regression` = 回归

In [ ]:
def mock_plan(question: str) -> QueryPlan:
    q = question.lower()
    metric = "cost" if "cost" in q else "message_count"
    if "channel" in q:
        dim = "channel"
    elif "use case" in q:
        dim = "use_case"
    else:
        dim = "month"
    return QueryPlan(metric=metric, dimension=dim)

golden = [
    ("Show monthly message trend", "message_count", "month"),
    ("Compare message volume by channel", "message_count", "channel"),
    ("Show cost by use case", "cost", "use_case"),
]

correct = 0
for q, expected_metric, expected_dim in golden:
    pred = mock_plan(q)
    ok = pred.metric == expected_metric and pred.dimension == expected_dim
    correct += int(ok)
    print(ok, q, pred.model_dump())

print("plan accuracy =", correct / len(golden))

### 练习：增加 Filter

把 QueryPlan 扩展为支持可选 `channel` filter，并保证 filter value 使用参数化 SQL，而不是字符串拼接。

<details>
<summary><b>提示</b></summary>

SQL 结构来自 allowlist；用户 value 用 `?` 参数。

</details>

<details>
    <summary><b>参考答案</b></summary>

    ```python
class QueryPlanV2(BaseModel):
    metric: Literal["message_count", "cost"]
    dimension: Literal["month", "channel", "use_case"]
    channel: Literal["SMS", "EMAIL", "PUSH"] | None = None
    limit: int = 100

def build_sql_v2(plan: QueryPlanV2):
    metric = METRICS[plan.metric]
    dim = DIMENSIONS[plan.dimension]
    params = []
    where = ""
    if plan.channel:
        where = "WHERE channel = ?"
        params.append(plan.channel)
    sql = f"SELECT {dim}, {metric} FROM messages {where} GROUP BY {dim} LIMIT ?"
    params.append(min(plan.limit, 100))
    return sql, tuple(params)
```

    </details>

### Checkpoint

1. 哪些事情应该让 LLM 决定？哪些不应该？
2. 为什么 metric formula 应放在 semantic layer？
3. Golden Case 最少应该测哪几个层次？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** LLM 可做意图理解、歧义处理、plan 建议；权限、允许字段、指标公式、row limit 等应由确定性程序控制。

**2.** 同一业务指标必须有统一定义，不能让模型每次临时解释/编公式。

**3.** 至少 intent/metric/dimension/filter/SQL execution；成熟后再加 answer/chart。

    </details>

## ChatBI 本周作业

        完成第一版真正 ChatBI：

`Question → QueryPlan → Safe SQL → PostgreSQL/SQLite → Table`

至少写 10 条 golden cases。

### 本周面试 / 理解检查

1. 为什么不推荐 LLM 直接执行任意 SQL？
2. Semantic Layer 的核心作用？
3. 为什么 SQL value 要参数化？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 安全、权限、字段幻觉、成本/大查询以及指标定义一致性都无法只靠模型保证。

**2.** 把业务 metric/dimension/关系/规则显式注册，作为 LLM 与物理数据库之间的受控语义契约。

**3.** 避免注入风险，并让数据库正确处理输入值。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。